In [3]:
import pandas as pd
import getpass
from huggingface_hub import notebook_login
import os

from datasets import load_dataset
from fireworks.client import Fireworks
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
from openai import OpenAI
import re


In [4]:
os.environ["FIREWORKS_API_KEY"] = getpass.getpass("fireworks api:")
client = Fireworks(api_key=os.environ["FIREWORKS_API_KEY"])

In [7]:
notebook_login()

In [8]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("openai api:")

openai_client = OpenAI(
    # This is the default and can be omitted
    api_key=os.environ.get("OPENAI_API_KEY"),
)

In [9]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

/home/markt/.local/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [10]:
instruction = """
          you are an expert translator between English and Danish

          #user will provide you sentences in English 
          #translate users english sentence to Danish.
          #you can only use users english sentence
          
  """


def translate_english_to_danish(english_samples_csv_file, model):
    list_of_danish_sentences = list()
    df = pd.read_csv(english_samples_csv_file)
    for i, row in enumerate(df.iterrows()):
        danish_sentence = client.chat.completions.create(
            model=model,
            messages=[
              {"role": "system", "content": instruction},
              {"role": "user", "content": row[1]['English']}
            ],
        )
        response = danish_sentence.choices[0].message.content
        list_of_danish_sentences.append(response)    
    return list_of_danish_sentences


generated_senetences_english_to_danish = translate_english_to_danish('./english-danish-openai.csv','accounts/fireworks/models/llama-v3-8b-instruct')


In [11]:
generated_senetences_english_to_danish

['Katten sad på vinduerneudsikten og så på fuglene udenfor.',
 "Hun bagte et skønt chokoladekage til hendes ven's fødselsdag.",
 'Nyt park i centrum af byen er blevet en populær destination for familier.',
 'Han realiserede sig hurtigt, at lære et nyt sprog kræver tålmodighed og øvelse.',
 'Slottet på bakken tilbyder en smuk udsigt over omgivelserne.',
 'De bestemte sig til at bruge deres feriemåneder på at udforske de fjerntliggende øer i Stillehavet.',
 'Fysikeren gjorde en banebrydende opdagelse, som kunne ændre fremtidens medicin.',
 'Børnene var opspekte over at starte deres første skoledag efter sommerferien.',
 'Hun fuldførte marathonen trods de udfordrende vejrforhold.',
 'Starten søger at skabe bæredygtige løsninger for byboerollen.',
 'Hanældre fortalte ham historier om de gamle dage, da landsbyen var meget mindre.',
 'Duften af nybrygget kaffe fylde rummet.',
 'Under stjernene på himlen lovede de at blive venner for altid.',
 'Trods det sygt regnvær fortsatte festivalen som 

In [12]:
instruction = """
        you are an expert translator between English and Danish

        #user will only give you samples of a sentence translated from English to Danish#
        # Give the translation a score on a scale from one to ten#
        #format should be something like 'score 1'#
        #give me an example how I can improve the instruction in order to improve score#
        #Think through your reasoning step-by-step and write the score in a new line at the end#
        score as int and placed as the last part#   
        #always write the score as '\nScore ' + the score - dont add other things!
              

          
  """


# instruction = """
#         you are an expert translator between English and Danish

#         #user will only give you samples of a sentence translated from English to Danish#
#         # Give the translation a score on a scale from one to ten#
#         #format should only be your score          
#   """


def evaluate_danish_sentences(english_danish_open_csv, evaluation_sentences):
    scores = list()
    df = pd.read_csv(english_danish_open_csv)
    df["Danish_llama_3"] = evaluation_sentences

    for i, row in enumerate(df.iterrows()): 
        
        response = openai_client.chat.completions.create(
            messages=[
                {"role": "system", "content": instruction},
                {"role": "user", "content": f'''English:{row[1]['English']} 
                                                Danish:{row[1]['Danish_llama_3']}'''}
            ],
            model="gpt-4o",

        )
        try:
            response = response.choices[0].message.content
            print(response + '\n\n' + '----------' + '\n\n')
            score = int(re.search("\d?\d", re.search("Score?:?\s\d?\d|\*?\*?\*?score?:?\*?\*\s+\d?\d", response)[0])[0])            
            scores.append(score)
        except json.JSONDecodeError as jde:
            continue

    return sum(scores) / len(scores)


llama_8b_avg_score = evaluate_danish_sentences("./english-danish-openai.csv", generated_senetences_english_to_danish)


First, let's analyze the Danish translation provided:

Original: The cat sat on the windowsill watching the birds outside.
Translation: Katten sad på vinduerneudsikten og så på fuglene udenfor.

Issues:
1. The word "vinduerneudsikten" is not correct Danish. The correct translation for "windowsill" is "vindueskarmen".
2. "Og så på" correctly means "and watched", but it can be streamlined to "kiggede på" for "watching".
3. The preposition "på" is used correctly here to denote "on" and "at".

Improved Translation: Katten sad på vindueskarmen og kiggede på fuglene udenfor.

Based on these observations:

Score 4

----------


The Danish translation has a couple of issues:

1. "Chokoladekage" should be "chokoladekage". The article "et" is correct, so no error here.
2. "Ven's" should be "vens" as there is no apostrophe in Danish possessive construction.
3. The word "hendes" is not needed; "sin" is more appropriate as it refers to the subject's own action directly.

An improved Danish translat

In [13]:
# print(f"Llama3 8B: {round(llama_8b_avg_score, 2)}")
print(llama_8b_avg_score)

5.949579831932773


In [19]:
# instruction_improved = """
#           you are an expert translator between English and Danish

#           #user will provide you sentences in English 
#           #translate users english sentence to Danish.
#           #following rules has to be followed, when you generate the danish sentences: 
#           # there is no apostrophe in Danish possessive construction
#           # windowsill is translated in danish 'vindueskarmen'
#           # Hanældre" should be "Hans bedstefar"
#           # "Han" (He) is missing between "Hans bedstefar" and "fortalte ham historier"
#           # "Sygt regnvær" is not commonly used in Danish; a better translation for "heavy rain" would be "kraftig regn" or "voldsomme regn."
#           #"Skjorter" means "shirts," not "scarves." The appropriate translation for "scarves" would be "halstørklæder."
#           #"Turejrejse" is not a common or accurate term in Danish. A better translation for "road trip" would be "biltur" or "roadtrip"
#           #"Camper" is understandable, but "gammel varevogn" would be more precise to describe an old van
#           #The Danish translation could be improved. "Kort" is a neuter noun, so it should be "et gammelt kort." Additionally, "loftet af" is not the optimal phrasing; "på loftet" would better convey the meaning of "in the attic."
#           #don't mention the rules in the translation
          
#   """

instruction_improved = """
          you are an expert translator between English and Danish

          #user will provide you sentences in Danish 
          #follow these following rules, when you generate the new danish sentences: 
          # there is no apostrophe in Danish possessive construction
          # windowsill is translated in danish 'vindueskarmen'
          # Hanældre" should be "Hans bedstefar"
          # "Han" (He) is missing between "Hans bedstefar" and "fortalte ham historier"
          # "Sygt regnvær" is not commonly used in Danish; a better translation for "heavy rain" would be "kraftig regn" or "voldsomme regn."
          #"Skjorter" means "shirts," not "scarves." The appropriate translation for "scarves" would be "halstørklæder."
          #"Turejrejse" is not a common or accurate term in Danish. A better translation for "road trip" would be "biltur" or "roadtrip"
          #"Camper" is understandable, but "gammel varevogn" would be more precise to describe an old van
          #The Danish translation could be improved. "Kort" is a neuter noun, so it should be "et gammelt kort." Additionally, "loftet af" is not the optimal phrasing; "på loftet" would better convey the meaning of "in the attic."
          #don't mention the rules in the translation
          
  """



def improved_danish_sentence_from_existing_list(english_danish_samples_csv_file, model):
    list_of_danish_sentences = list()
    df = pd.read_csv(english_danish_samples_csv_file)
    for i, row in enumerate(df.iterrows()):
        danish_sentence = client.chat.completions.create(
            model=model,
            messages=[
              {"role": "system", "content": instruction_improved},
              {"role": "user", "content": row[1]['Danish']}
            ],
        )
        response = danish_sentence.choices[0].message.content
        list_of_danish_sentences.append(response)    
    return list_of_danish_sentences


generated_senetences_english_to_danish = translate_english_to_danish('./english-danish-openai.csv','accounts/fireworks/models/llama-v3-8b-instruct')

generated_senetences_english_to_danish

['Katten sat på vindueskarmen og betragtede fuglene udenfor.',
 'Hun bagede en lækker kakao-tærte til hendes venins fødselsdag.',
 'Byen i centret har nu et nyt park, der er blevet en populær spot for familier.',
 'Han kom hurtigt til at forstå, at læring af et nyt sprog kræver tålmodighed og øvelse.',
 'Slottet på bakken tilbyr et smukt udsyn over landsbyen.',
 'De bestemte sig for at tilbringe ferien udforskning Islands i det Sydlige Stillehav.',
 'Forskeren gjorde en revolutionært opdagelse, der kunne ændre fremtidens medicinske udvikling.',
 'Barnet var spændt afsløreldag i skolen efter sommerferien.',
 'Hun fuldførte marathonen trods de vanskelige vejrforhold.',
 'Nyhudsbedriften arbejder på at skabe bæredygtige løsninger for bymæssigt liv.',
 'Hans bedstefar fortalte ham historier om de gamle dage, da landsbyen var meget mindre.',
 'Ru Omni om byen kaffe fyldte stuen.',
 'Under stjernene i himlen lovobrejde de at blive venner for evigt.',
 'Trosset mod kraftig regn fortsatte fest

In [25]:

# def evaluate_danish_sentences(english_danish_open_csv, evaluation_sentences):
#     scores = list()
#     df = pd.read_csv(english_danish_open_csv)
#     df["Danish_llama_3"] = evaluation_sentences

#     for i, row in enumerate(df.iterrows()): 
        
#         response = openai_client.chat.completions.create(
#             messages=[
#                 {"role": "system", "content": instruction},
#                 {"role": "user", "content": f'''English:{row[1]['English']} 
#                                                 Danish:{row[1]['Danish_llama_3']}'''}
#             ],
#             model="gpt-4o",

#         )
#         try:
#             response = response.choices[0].message.content
#             print(response + '\n\n' + '----------' + '\n\n')
#             matches =  re.search("Score?:?\s\d?\d|\*?\*?\*?score?:?\*?\*\s+\d?\d", response)
            
#             if any(re.search(pattern, matches) for pattern in "\d?\d"):
#                 score = int(matches[0])
#                 scores.append(score)
#         except json.JSONDecodeError as jde:
#             continue

#     return sum(scores) / len(scores)


llama_8b_avg_score = evaluate_danish_sentences("./english-danish-openai.csv", generated_senetences_english_to_danish)

Score 9

----------




error: bad escape (end of pattern) at position 0

In [ ]:
print(llama_8b_avg_score)